# Setup Dataset e Preparazione per ResNet

## 1. Import e configurazioni di setup

In [ ]:
import os
import sys
import glob # modulo per trovare file che corrispondono a un pattern (esempio: *.jpeg)
from pathlib import Path # per gestire meglio i file al posto delle stringhe

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

import matplotlib.pyplot as plot
import seaborn as sns
# per visualizzare i grafici direttamente nel notebook
%matplotlib inline 

from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import DataLoader # classe che carica i dati in batch per il training
from torchvision import transforms, models

from tqdm import tqdm # mi serve così visualizzo meglio l'avanzamento quando questo è troppo lungo

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
ROOT = Path('..')
BATCH_SIZE = 32 
IMG_SIZE = 224
N_CLASSES = 5  # Apple, Banana, Grape, Mango, Strawberry
# dimensioni standard

## 2. Normalizzazione e trasformazioni
Servono a 

In [2]:
NORMALIZE = transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])

TRAIN_TRANSFORMS = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.AutoAugment(transforms.AutoAugmentPolicy.IMAGENET),  # Augmentation automatica
    transforms.ToTensor(),
    NORMALIZE
])

VAL_TEST_TRANSFORMS = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    NORMALIZE
])

## 3. Verifica Dataset Scaricato

Il dataset è già pre-diviso in:
- **train/** (~1900-1940 immagini per classe)
- **valid/** (~39-40 immagini per classe)
- **test/** (~19-20 immagini per classe)

Totale: ~2000 immagini per classe (perfettamente bilanciato)

In [3]:
dataset_path = ROOT / "data" / "fruits-classification"

if dataset_path.exists():
    print(f"Dataset trovato in: {dataset_path}\n")
    
    # Conta le immagini per classe in ogni split
    class_data = {}
    
    for split in ['train', 'valid', 'test']:
        split_path = dataset_path / split
        print(f"{split.upper()}:")
        
        for class_name in sorted(os.listdir(split_path)):
            class_folder = split_path / class_name
            if class_folder.is_dir():
                n_images = len(list(class_folder.glob("*.jpeg")))
                
                if class_name not in class_data:
                    class_data[class_name] = {}
                class_data[class_name][split] = n_images
                print(f"  {class_name}: {n_images} immagini")
    
    # Stampa il totale per ogni classe
    print("\nTOTALE PER CLASSE:")
    for class_name in sorted(class_data.keys()): #keys è un metodo che restituisce le chiavi di un dizionario
        total = sum(class_data[class_name].values())
        print(f"  {class_name}: {total} immagini")
    
    print("\nDataset bilanciato")
else:
    print(f"Dataset non trovato in {dataset_path}")

Dataset trovato in: ../data/fruits-classification

TRAIN:
  Apple: 1923 immagini
  Banana: 1935 immagini
  Grape: 1940 immagini
  Mango: 1940 immagini
  Strawberry: 1940 immagini
VALID:
  Apple: 39 immagini
  Banana: 40 immagini
  Grape: 40 immagini
  Mango: 40 immagini
  Strawberry: 40 immagini
TEST:
  Apple: 19 immagini
  Banana: 20 immagini
  Grape: 20 immagini
  Mango: 20 immagini
  Strawberry: 20 immagini

TOTALE PER CLASSE:
  Apple: 1981 immagini
  Banana: 1995 immagini
  Grape: 2000 immagini
  Mango: 2000 immagini
  Strawberry: 2000 immagini

Dataset bilanciato


## 4. Verifica stratificazione
Verifico che il dataset sia stratificato nelle percentuali 80%, 10%, 10%

In [8]:

# Conta le immagini in ogni split
split_counts = {}

for split in ['train', 'valid', 'test']:
    split_path = dataset_path / split
    n_images = len(list(split_path.glob("*/*.jpeg")))
    split_counts[split] = n_images

total_images = sum(split_counts.values())
train_perc = (split_counts['train'] / total_images) * 100
valid_perc = (split_counts['valid'] / total_images) * 100
test_perc = (split_counts['test'] / total_images) * 100

print(f"Train: {split_counts['train']} immagini ({train_perc:.1f}%)")
print(f"Valid: {split_counts['valid']} immagini ({valid_perc:.1f}%)")
print(f"Test: {split_counts['test']} immagini ({test_perc:.1f}%)")
print(f"Total: {total_images} immagini\n")

# Se è già 80/10/10, va bene cosi, altrimenti redistribuisco
# La verifica la faccio con una tolleranza dell'1% (standard per dataset di piccole dimensioni)
if 79 <= train_perc <= 81 and 9 <= valid_perc <= 11 and 9 <= test_perc <= 11:
    print("Dataset stratificato: 80%, 10%, 10%")
else:
    print("Dataset non stratificato: eseguo ridistribuzione")
    
    # Leggi tutti i percorsi e le label da tutte le immagini
    all_paths = list(dataset_path.glob("*/*/*.jpeg"))  # attenzione ai livelli delle cartelle
    all_classes = [p.parent.name for p in all_paths]  
    
    # Primo split 80/20
    train_paths, temp_paths, train_classes, temp_classes = train_test_split(
        all_paths, all_classes, test_size=0.2, random_state=RANDOM_SEED, stratify=all_classes
    )
    
    # Secondo split 20/20 = 10% e 10%
    valid_paths, test_paths, valid_classes, test_classes = train_test_split(
        temp_paths, temp_classes, test_size=0.5, random_state=RANDOM_SEED, stratify=temp_classes
    )
    
    print(f"  Train: {len(train_paths)} immagini ({len(train_paths)/total_images*100:.1f}%)")
    print(f"  Valid: {len(valid_paths)} immagini ({len(valid_paths)/total_images*100:.1f}%)")
    print(f"  Test: {len(test_paths)} immagini ({len(test_paths)/total_images*100:.1f}%)")
    
    # Salva il dataset redistribuito
    import shutil
    new_dataset_path = ROOT / "data" / "fruits-classification-stratified"
    new_dataset_path.mkdir(parents=True, exist_ok=True)
    
    for split_name, paths_list, labels_list in [
        ('train', train_paths, train_classes),
        ('valid', valid_paths, valid_classes),
        ('test', test_paths, test_classes)
    ]:
        split_dir = new_dataset_path / split_name
        split_dir.mkdir(exist_ok=True)
        
        for img_path, class_name in zip(paths_list, labels_list):
            class_dir = split_dir / class_name
            class_dir.mkdir(exist_ok=True)
            shutil.copy(str(img_path), str(class_dir / img_path.name))
    
    dataset_path = new_dataset_path

Train: 9678 immagini (97.0%)
Valid: 199 immagini (2.0%)
Test: 99 immagini (1.0%)
Total: 9976 immagini

Dataset non stratificato: eseguo ridistribuzione
  Train: 7980 immagini (80.0%)
  Valid: 998 immagini (10.0%)
  Test: 998 immagini (10.0%)


## 5. DataLoader 
### Preparazione dei Dati per la Rete Neurale

I DataLoader sono strumenti che si occupano di:

1. Creazione di batch: Le immagini non vengono caricate tutte insieme, ma suddivise in pacchetti (batch) per evitare problemi di memoria
2. Mescolamento: Le immagini vengono mescolate casualmente durante il training, così la rete neurale apprende realmente e non impara solo l'ordine (così evito l'overfitting), metre niente mescolamento per validation e test (mi servono immagini pulite e risproducibilità)
3. Trasformazioni: Conversione delle immagini in tensori matematici, normalizzazione, augmentation (già definite nella cella 2)

In [9]:
from torchvision.datasets import ImageFolder

# Crea i dataset direttamente dalle cartelle (molto piu comodo)
train_dataset = ImageFolder(root=dataset_path / 'train', transform=TRAIN_TRANSFORMS)
val_dataset = ImageFolder(root=dataset_path / 'valid', transform=VAL_TEST_TRANSFORMS)
test_dataset = ImageFolder(root=dataset_path / 'test', transform=VAL_TEST_TRANSFORMS)

# Crea i DataLoader con specifiche di batch size, shuffle e num_workers standard
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=4)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=4)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=4)

print("Dataset e DataLoader creati")
print(f"Train: {len(train_dataset)} immagini, {len(train_loader)} batch")
print(f"Validation: {len(val_dataset)} immagini, {len(val_loader)} batch")
print(f"Test: {len(test_dataset)} immagini, {len(test_loader)} batch")

Dataset e DataLoader creati
Train: 7980 immagini, 250 batch
Validation: 998 immagini, 32 batch
Test: 998 immagini, 32 batch


# ResNet18 e Fine Tuning

## 1. Setup e Import di ResNet

### 1.1. Importazione ResNet18

**ResNet18** è una rete neurale convoluzionale (CNN) già addestrata su 1 milione di immagini (dataset ImageNet), ha 18 layer (strati), pesa circa 45MB e normalmente classifica 1000 categorie (perché ImageNet ne ha 1000)

In [ ]:
# Importazione di ResNet18 da torchvision
# pretrained=True scarica i pesi da ImageNet (45MB la prima volta, poi cache locale)
model = models.resnet18(pretrained=True)

print("ResNet18 importato e caricato con pesi pre-addestrati")

# Spostare il modello sul device (GPU o CPU)
model = model.to(device)

print(f"Modello spostato su: {device}")

### 1.2. Architettura ResNet18

**Struttura:**
- Layer convoluzionali (estraggono features dalle immagini)
- Layer batch normalization
- Layer ReLU (attivazione)
- Ultimo layer FC (fully connected) con 1000 output

L'ultimo layer verrà poi cambiato da 1000 a 5 (le classi del dataset)

In [ ]:
print("Struttura di ResNet18:")
print("="*60)
print(model)
print("="*60)

### 1.3. Parametri del Modello

I **parametri** sono i pesi della rete neurale
Ogni neurone ha pesi che si aggiustano durante l'allenamento
(ResNet18 circa 11.7 milioni di parametri)

In [ ]:
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Parametri totali: {total_params:,}")
print(f"Parametri addestrabili: {trainable_params:,}")
print(f"\nDimensione approssimativa in memoria: {total_params * 4 / 1024 / 1024:.2f} MB")
print("(ogni parametro occupa ~4 byte)")

### 1.4. Check ultimo Layer

ResNet18 finisce con un layer FC (fully connected) che ha 1000 output

Importante controllare quanti input features ha questo layer (512 per ResNet18) e quanti output ha (1000 - ImageNet)

In [ ]:
print("Ultimo layer (Fully Connected):")
print(f"  Nome: {list(model.named_modules())[-1][0]}")
print(f"  Tipo: {type(model.fc)}")
print(f"\nDettagli del layer FC:")
print(f"  Input features: {model.fc.in_features}")
print(f"  Output features (classi ImageNet): {model.fc.out_features}")
print(f"\nNoi dobbiamo cambiarli in: {N_CLASSES} (le nostre 5 classi di frutta)")